In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
w = 5
h = 5


In [ ]:
bbox_pts = np.array([[0, 0], [w, 0], [w, h], [0, h]])
bbox_edges = [[0, 1], [1, 2], [2, 3], [3, 0]]

In [ ]:
triArea = 0.01

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(bbox_pts, bbox_edges, triArea)


In [ ]:
marker = [False] * m.numVertices()

In [ ]:
fuse_boundary = True

# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            marker[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            marker[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            marker[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            marker[i] = True    



In [ ]:
isheet = inflation.InflatableSheet(m, marker)

In [ ]:
isheet.thickness *= h / 5

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
fixedVars, hessianShift = [], 1e-6



In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 0.1
opts.niter = 2000
opts.gradTol = 1e-10
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update()
cr = inflation.inflation_newton(isheet, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(isheet), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10, fixedVars = fixedVars)


In [ ]:

import mode_viewer, importlib
mview = mode_viewer.ModeViewer(isheet, modes, lambdas, amplitude=50)
mview.show()